<center style="padding:1rem 0;">
    <h1 style="font-size: 4rem;">IA - Deep Learning</h1>
    <h2 style="font-size: 2rem;">Livrable 2 - Construction d'un premier réseau de neurones</h2>
    <h5 style="font-size: 1rem;"><i>Thomas VINET, Hugo HELM, Alban GODIER</i></h5>
</center>

<img src="assets/cesi.png" style="position:absolute;right:2rem;top:4.5rem;width:10rem;background:#fee237;"/>

In [1]:
from __future__ import annotations
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.neural_network import NeuralNetwork, DrawRealTimeLoss, EarlyStopping

ImportError: cannot import name 'NeuralNetwork' from partially initialized module 'lib.neural_network' (most likely due to a circular import) (/home/jovyan/notebook/lib/neural_network/__init__.py)

## Descente de gradient

### Introduction

Lors de la descente de gradient dans note modèle, nous cherchons à ajuster les poids et les biais de chaque couche du réseau de neurone. Pour ce faire, nous partons de la sortie du réseau pour chercher à optimiser le résultat de la fonction de coût. 

On considère un réseau de neurones entièrement connecté, composé de couches indexées par $i \in \{0, \dots, L\}$ où :
- $i = 0$ : couche de sortie
- $i = L$ : couche d’entrée

On pose :
- $y$ : la valeur attendue pour la prédiction
- $\hat y$ : la prédiction du modèle
- $\mathcal L$ la fonction de coût du modèle, définie en fonction de $y$ et $\hat y$
- $A_i$ la fonction d'activation de la couche $i$, appliquée composant par composant
- $z_{i}$ la matrice des sorties des fonctions d'agrégation des neurones de la couche $i$
- $x_i$ la matrice des entrées pour les neurones de la couche $i$
- $w_{i}$ la matrice des poids liés aux entrées $x_i$ pour les neurone de la couche $i$
- $b_{i}$ la matrice des biais des neurones de la couche $i$

Les dimensions des matrices sont les suivantes :
- $x_L$ : $\text{variables entrée} \times 1$ , la matrice a une ligne par variable d'entrée (sortie de la couche $i+1$) et 1 colonne.
- $w_i$ : $\text{Neurones}_i \times \text{Neurones}_{i+1}$ , la matrice a une ligne par neurone de la couche $i$ et une colonne par neurone de la couche $i+1$
- $b_i$ : $\text{Neurones}_i \times 1$ , la matrice a une ligne par neurone de la couche $i$ et 1 colonne.

Pour chaque couche du réseau :
$$
\begin{cases}
z_i = w_ix_i + b_i \\
x_{i-1} = A_i(z_i)
\end{cases}
$$

On note également la prédiction du modèle et la fonction de coût :
- $\hat y = A_0(z_0)$
- $\mathcal L = \mathcal L(y, \hat y)$

Dans le cas de la descente de gradient, on cherche à connaitre :
$$\frac{\partial \mathcal L}{\partial w_i} \quad et \quad \frac{\partial \mathcal L}{\partial b_i}$$
Pour cela, on pose la variable :
$$\delta_i = \frac{\partial \mathcal L}{\partial z_i}$$

Nous cherchons donc à définir tous les gradients des poids et des biais en fonction de ces valeurs.
Afin d'avoir une formule plus dynamique pour le modèle, nous voulons obtenir des expressions récurrentes, en afin d'obtenir le gradient d'une couche $i$ en fonction de la couche précédente ($i-1$)

### Exemple couche 0

Afin de démontrer notre relation récursive, on commence par définir le gradient de la dernière couche (couche de sortie).
On a :
- $\hat y = A_0(z_0)$, $A_0$ étant la fonction d'activation de la dernière couche
- $z_0=w_0x_0+b_0$

On cherche : $\frac{\partial \mathcal L}{\partial w_0}$ et $\frac{\partial \mathcal L}{\partial b_0}$
Pour cela on cherche $\delta_0$ afin de les obtenir par composition.


On pose :
$$
\begin{align}
\delta_0 &= \frac{\partial \mathcal L}{\partial \hat y} \cdot \frac{\partial \hat y}{\partial z_0} \\
&= \frac{\partial \mathcal L}{\partial \hat y} \odot A'_0(z_0)
\end{align}
$$
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_0}
	& =
	\delta_0 \cdot \frac{\partial z_0}{\partial w_0} \\
	& =
	\delta_0 \cdot x_0
\end{align}
$$
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_0}
	& =
	\delta_0 \cdot \frac{\partial z_0}{\partial b_0} \\
	& =
	\delta_0
\end{align}
$$

### Exemple couche 1

On définit ensuite le gradient de notre première couche cachée (couche 1).
On a :
- $z_0=w_0 x_0 + b_0$
- $x_0=A_1(z_1)$
- $z_1=w_1 x_1 + b_1$

On cherche : $\frac{\partial \mathcal L}{\partial w_1}$ et $\frac{\partial \mathcal L}{\partial b_1}$
Pour cela on cherche $\delta_1$, comme pour la dernière couche.
On pose :
$$
\begin{align}
	\delta_1
	& =
	\frac
	 {\partial \mathcal L}
	 {\partial \hat y} 
	\cdot 
	\frac
	 {\partial \hat y}
	 {\partial z_0}
	\cdot
	\frac
	 {\partial z_0}
	 {\partial x_0}
	\cdot
	\frac
	 {\partial x_0}
	 {\partial z_1} \\
	& =
	(\delta_0
	\cdot
	w_0)
	\odot
	A_1'(z_1)
\end{align}
$$
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_1}
	& =
	\delta_1 \cdot \frac{\partial z_1}{\partial w_1}  \\
	& =
	\delta_1 \cdot x_1
\end{align}
$$
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_1}
	& =
	\delta_1 \cdot \frac{\partial z_1}{\partial b_1}  \\
	& =
	\delta_1
\end{align}
$$

### Relation récursive

En partant de l'expression de la couche 1 :
$$
\begin{align}
	\delta_1
	& =
	(\delta_0
	\cdot
	w_0)
	\odot
	A_1'(z_1)
\end{align}
$$
En prenant $i=1$, on peut définir la relation suivante :
$$
\begin{align}
	\delta_i
	& =
	(\delta_{i-1}
	\cdot
	w_{i-1})
	\odot
	A_i'(z_i)
\end{align}
$$

### Dérivée des fonctions de coût

#### Cross-entropy (cross-entropy)

$$
    \mathcal L
    =
    -y \log(\hat y) - (1- y) \log(1- \hat y)
    $$
    $$
    \frac{\partial \mathcal L}{\partial \hat y}
    =
    -\frac{y}{\hat y} - \frac{1-y}{1 - \hat y}
$$

#### Mean squared error (MSE)

$$
    MSE 
    = 
    \frac{1}{n}
    \sum^n_{j=1}{(y_j - \hat y_j)^2}
    $$
    $$
    \frac{\partial \mathcal L}{\partial \hat y_j}
    =
    \frac{2}{n} (\hat y_j - y_j)
$$

#### Mean absolute error (MAE)

$$
    MAE 
    = 
    \frac{1}{n}
    \sum^n_{j=1}{|y_j - \hat y_j|}
    $$
    $$
    \frac{\partial\mathcal L}{\partial \hat y_i}
    =
    \begin{cases}
     \begin{align}
      & 1 & \text{pour}\ \hat y_i > y_i \\ 
      & -1 & \text{pour}\ \hat y_i < y_i\\
      & \text{indéfini} & \text{pour}\ \hat y_j = y_j
      \end{align}
    \end{cases}
$$

### Dérivée des fonctions d'activation

#### Sigmoïde

$$
    \sigma(x) 
    =
    \frac{1}{1+e^{-x}}
    $$
    $$
    \sigma'(x)
    =
    \frac{e^{-x}}{(1+e^{-x})^2}
    =
    \sigma(x)(1-\sigma(x))
$$
#### ReLU
$$
    a(x)=\max(0,x)
$$
$$
    a'(x)=
    \begin{cases}
     \begin{align}
      & 0 & \text{pour}\ x<0 \\
      & 1 & \text{pour}\ x\ge 0
     \end{align}
    \end{cases}
$$
#### Tanh
$$
    a(x)
    =
    \tanh(x)
    $$
    $$
    a'(x)
    =
    \frac
     {1}
     {\cosh(x)^2}
    =
    1-\tanh(x)^2
$$
